In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [3]:
!pip install faiss-cpu wikipedia


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Wikipedia Retriver

In [4]:
from langchain_community.retrievers import WikipediaRetriever


In [5]:
# Initialize the retriever
retriever = WikipediaRetriever(top_k_results=2 , lang="en")

In [6]:
# Definining Query
query = "Tell me about the conflict between Russia and Ukraine"

# Get relevant wikipedia docs
docs = retriever.invoke(query)

In [7]:
docs

[Document(metadata={'title': 'Peace negotiations in the Russo-Ukrainian war (2022–present)', 'summary': 'There have been several rounds of peace talks to end the ongoing Russo-Ukrainian war since it began with Russia\'s invasion in February 2022. Russia\'s president Vladimir Putin seeks recognition of all occupied land as Russian, for Russia to be given all of the regions it claims but does not fully occupy, guarantees that Ukraine will never join NATO, curtailment of Ukraine\'s military, and the lifting of sanctions against Russia. Ukraine\'s president Volodymyr Zelenskyy seeks a ceasefire, full withdrawal of Russian troops, the return of prisoners and kidnapped Ukrainian children, prosecution of Russian leaders for war crimes, and security guarantees to prevent further Russian aggression.\nThe first meeting between Russian and Ukrainian officials was held four days after the invasion began, on 28 February 2022, in Belarus, and ended without result. Later rounds of talks took place in

In [8]:
# Print retrieved content 

for i , doc in enumerate(docs):
    print(f"\n--- Result{i+1} ---")
    print(f"Content: \n {doc.page_content}... ") # truncate for display
    



--- Result1 ---
Content: 
 There have been several rounds of peace talks to end the ongoing Russo-Ukrainian war since it began with Russia's invasion in February 2022. Russia's president Vladimir Putin seeks recognition of all occupied land as Russian, for Russia to be given all of the regions it claims but does not fully occupy, guarantees that Ukraine will never join NATO, curtailment of Ukraine's military, and the lifting of sanctions against Russia. Ukraine's president Volodymyr Zelenskyy seeks a ceasefire, full withdrawal of Russian troops, the return of prisoners and kidnapped Ukrainian children, prosecution of Russian leaders for war crimes, and security guarantees to prevent further Russian aggression.
The first meeting between Russian and Ukrainian officials was held four days after the invasion began, on 28 February 2022, in Belarus, and ended without result. Later rounds of talks took place in March 2022 on the Belarus–Ukraine border and in Antalya, Turkey. Negotiations in 

Vector Store Retriever

In [17]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings , HuggingFaceEndpoint , ChatHuggingFace
from langchain_core.documents import Document

In [18]:
# Step 1:source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [22]:
# Step 2: Initialize embedding models 
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 3 : Create chroma vector store in a memory
vectorstore = Chroma.from_documents(
    documents = documents ,
    embedding = embedding_model ,
    collection_name = "my_collection"
)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
#Step 4 : Convert vectorstore as retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [24]:
query = "What is Chroma used for ?"
results = retriever.invoke(query)

In [25]:
for i , doc in enumerate(results):
    print(f"\n--Result {i+1}--")
    print(doc.page_content)


--Result 1--
Chroma is a vector database optimized for LLM-based search.

--Result 2--
LangChain helps developers build LLM applications easily.


MMR = Maximum MarginaL Relevance


In [27]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [28]:
from langchain_community.vectorstores import FAISS

# Step 1: Initialize embeddings model 
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
# Enable MMR as a retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [30]:
query = "What is Langchain?"
results = retriever.invoke(query)

In [31]:
for i , doc in enumerate(results):
    print(f"\n --Result {i+1} --")
    print(doc.page_content)


 --Result 1 --
LangChain supports Chroma, FAISS, Pinecone, and more.

 --Result 2 --
LangChain is used to build LLM based applications.

 --Result 3 --
Embeddings are vector representations of text.


Multi-Query Retriever

In [38]:
pip install langchain-classic


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever

In [41]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [43]:
#Initialize Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs , embedding = embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [44]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type = 'similarity' , search_kwargs = {"k":5})



In [46]:
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace


In [66]:
mutliquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs ={"k":5}) ,
    llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-72B-Instruct" ,
    task = "text-generation"
))



In [62]:
# Query 
query = "How to improve energy levels and maintain balance?"

In [ ]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
mutliquery_results = mutliquery_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)